In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd().resolve()
if package_dir.name == 'examples':
    package_dir = package_dir.parent
else:
    repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])

0

In [ ]:
import importlib
import ce_visualization_plotly.plugin as plotly_plugin

importlib.reload(plotly_plugin)
plotly_plugin.register_plotly_visualization_components()

# Standalone Plotly instance workspace

This notebook demonstrates Mode A: standalone HTML with precomputed local cards.

Standalone mode creates a shareable HTML dashboard. The exported file can inspect the global instance explorer and any factual or alternative local cards that were precomputed before export. It cannot generate new factual or alternative explanations after the HTML file has been written; on-demand generation requires live dashboard mode.

In [8]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from calibrated_explanations import WrapCalibratedExplainer

# Importing the package registers its Plotly styles with calibrated-explanations.
import ce_visualization_plotly  # noqa: F401

## Data

We use a small deterministic binary classification problem and split it into proper training, calibration, and query/test sets using a 60/20/20 split.

In [9]:
x, y = make_classification(
    n_samples=300,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    random_state=0,
)

x_train, x_query, y_train, y_query = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=0,
    stratify=y,
)
x_proper, x_cal, y_proper, y_cal = train_test_split(
    x_train,
    y_train,
    test_size=0.25,
    random_state=0,
    stratify=y_train,
)

x_proper.shape, x_cal.shape, x_query.shape

((180, 8), (60, 8), (60, 8))

## Fit and calibrate

The wrapper is fitted on the proper training split and calibrated on the calibration split. The query data is held out for the dashboard.

In [10]:
model = RandomForestClassifier(n_estimators=100, random_state=0)
explainer = WrapCalibratedExplainer(model)

explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

## Optional sanity plot

The standalone dashboard starts from the same global instance explorer shown below.

In [11]:
global_result = explainer.plot(
    x_query,
    y_query,
    style="plotly.global.instance_explorer",
    task="classification",
    position_precision=2,
    show=True,
)

## Export a standalone dashboard

This exports `dashboard_instance_workspace_standalone.html`. The dashboard precomputes cards for the most uncertain query instances, so those cards remain available inside the static HTML file. Instances that were not precomputed can still be inspected in the global summary, but they cannot ask Python for new explanations after export.

In [ ]:
result = explainer.plot(
    x_query,
    y_query,
    style="plotly.dashboard.instance_workspace",
    dashboard_mode="standalone_html",
    precompute="top_uncertain",
    max_precomputed_instances=10,
    available_cards="auto",
    include_factual=True,
    include_alternatives=True,
    include_conjunctions=True,
    max_rule_size=3,
    path="dashboard_instance_workspace_standalone.html",
    show=True,
)

result.saved_paths

TypeError: VennAbers.predict_proba() got an unexpected keyword argument 'style'

## Underlying cards

The dashboard uses only the Plotly cards currently implemented in this package: `plotly.local.uncertainty_quadrant`, `plotly.local.ensured`, and `plotly.local.alternative_feature_summary`. The global overview is `plotly.global.instance_explorer`.